# 🧪 Lab 7: The Photon Torpedo Suite (SQL Tab Optimization & Plan Forensics)

In this laboratory session, we audit Spark SQL's physical execution layers to trace logical-to-physical transformations, inspect Adaptive Query Execution (AQE) behavior, compare native Spark expressions with Python UDFs, and trigger an intentional ANSI failure.

**Objective:** Learn how structured workloads appear in the Spark SQL tab: physical plans, exchanges, joins, operator metrics, AQE evidence, Python worker transfer metrics, and failed-query diagnostics.


### Step 1: Initialize the Tactical Target Suite

We start a local Spark session, explicitly enable the Spark Web UI, AQE, and ANSI mode, and print the actual UI URL.

Do not assume port `4040`. Spark requests it, but if the port is busy it may retry on another port. Trust `spark.sparkContext.uiWebUrl`.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
import time

existing = SparkSession.getActiveSession()
if existing is not None:
    existing.stop()

spark = (
    SparkSession.builder
    .appName("spark-ui-sql-photon-torpedoes-lab-07")
    .master("local[*]")
    .config("spark.ui.enabled", "true")
    .config("spark.ui.port", "4040")
    .config("spark.port.maxRetries", "16")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.ansi.enabled", "true")
    .config("spark.sql.shuffle.partitions", "100")
    .getOrCreate()
)

print(f"Active Spark version: {spark.version}")
print(f"🛰️ Actual Spark UI: {spark.sparkContext.uiWebUrl}")
print(f"AQE enabled: {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"ANSI mode enabled: {spark.conf.get('spark.sql.ansi.enabled')}")
print(f"Shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")


Active Spark version: 4.1.2
🛰️ Actual Spark UI: http://T14-PF4WM3XL.sdggroup.local:4040
AQE enabled: true
ANSI mode enabled: true
Shuffle partitions: 100


### Step 2: Inspect AQE on a Shuffle Workload

We create a structured aggregation with many configured shuffle partitions and a selective filter before the aggregation.

With AQE enabled, Spark may use runtime statistics to coalesce post-shuffle partitions. Verify the actual behavior in the SQL tab or the final adaptive physical plan. Do not assume coalescing happened just because AQE is enabled.


In [2]:
spark.sparkContext.setJobDescription("🛡️ Target Alpha: SQL Aggregation and AQE Inspection")

raw_matrix = (
    spark.range(0, 5_000_000, 1, numPartitions=64)
    .withColumn("sector_id", (F.col("id") % 1000).cast("int"))
    .filter(F.col("id") < 50_000)
)

coalesced_aggregation = raw_matrix.groupBy("sector_id").count()

print("🧱 Pre-flight query plan mapping:")
coalesced_aggregation.explain("formatted")

start_clock = time.time()
aqed_rows = coalesced_aggregation.collect()
end_clock = time.time()

print(f"🎯 Target Alpha materialized: {len(aqed_rows)} grouped rows.")
print(f"⏱️ Duration: {end_clock - start_clock:.2f} seconds.")
print("Open the SQL tab and inspect the final/adaptive plan for exchanges, query stages, and possible coalesced shuffle readers.")


🧱 Pre-flight query plan mapping:
== Physical Plan ==
AdaptiveSparkPlan (7)
+- HashAggregate (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Project (3)
            +- Filter (2)
               +- Range (1)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 5000000, step=1, splits=Some(64))

(2) Filter
Input [1]: [id#0L]
Condition : (id#0L < 50000)

(3) Project
Output [1]: [cast((id#0L % 1000) as int) AS sector_id#1]
Input [1]: [id#0L]

(4) HashAggregate
Input [1]: [sector_id#1]
Keys [1]: [sector_id#1]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#6L]
Results [2]: [sector_id#1, count#7L]

(5) Exchange
Input [2]: [sector_id#1, count#7L]
Arguments: hashpartitioning(sector_id#1, 100), ENSURE_REQUIREMENTS, [plan_id=19]

(6) HashAggregate
Input [2]: [sector_id#1, count#7L]
Keys [1]: [sector_id#1]
Functions [1]: [count(1)]
Aggregate Attributes [1]: [count(1)#5L]
Results [2]: [sector_id#1, count(1)#5L AS count#2L]

(7) AdaptiveSparkPlan
Output [2]: [sector_

### Step 3: Trigger a Controlled Row Explosion

We create a many-to-many join where each side has repeated keys. The output is much larger than either input.

In the SQL tab, inspect the join operator and the `number of output rows` metric. This is where a harmless-looking join can reveal the tribble reproduction program hiding under your DataFrame code.


In [3]:
spark.sparkContext.setJobDescription("💥 Target Beta: Controlled Join Row Explosion")

left_fleet = (
    spark.range(0, 20_000, 1, numPartitions=16)
    .withColumn("join_key", (F.col("id") % 100).cast("int"))
    .withColumnRenamed("id", "left_id")
)

right_fleet = (
    spark.range(0, 20_000, 1, numPartitions=16)
    .withColumn("join_key", (F.col("id") % 100).cast("int"))
    .withColumnRenamed("id", "right_id")
)

row_explosion = left_fleet.join(right_fleet, on="join_key", how="inner")

print("🧱 Join plan mapping:")
row_explosion.explain("formatted")

start_clock = time.time()
exploded_count = row_explosion.count()
end_clock = time.time()

print(f"🎯 Target Beta produced {exploded_count:,} joined rows.")
print(f"⏱️ Duration: {end_clock - start_clock:.2f} seconds.")
print("Open the SQL tab and look for join output rows, exchanges, broadcast/shuffle strategy, and associated jobs/stages.")


🧱 Join plan mapping:
== Physical Plan ==
AdaptiveSparkPlan (10)
+- Project (9)
   +- BroadcastHashJoin Inner BuildRight (8)
      :- Project (3)
      :  +- Filter (2)
      :     +- Range (1)
      +- BroadcastExchange (7)
         +- Project (6)
            +- Filter (5)
               +- Range (4)


(1) Range
Output [1]: [id#8L]
Arguments: Range (0, 20000, step=1, splits=Some(16))

(2) Filter
Input [1]: [id#8L]
Condition : isnotnull(cast((id#8L % 100) as int))

(3) Project
Output [2]: [id#8L AS left_id#10L, cast((id#8L % 100) as int) AS join_key#9]
Input [1]: [id#8L]

(4) Range
Output [1]: [id#11L]
Arguments: Range (0, 20000, step=1, splits=Some(16))

(5) Filter
Input [1]: [id#11L]
Condition : isnotnull(cast((id#11L % 100) as int))

(6) Project
Output [2]: [id#11L AS right_id#13L, cast((id#11L % 100) as int) AS join_key#12]
Input [1]: [id#11L]

(7) BroadcastExchange
Input [2]: [right_id#13L, join_key#12]
Arguments: HashedRelationBroadcastMode(List(cast(input[1, int, true] as bigint)

### Step 4: Compare Native Expressions with a Python UDF Boundary

First we classify rows using native Spark SQL expressions. Then we repeat similar logic with a row-at-a-time Python UDF.

The native path should stay inside Spark's optimized SQL engine. The Python UDF path should introduce Python execution operators such as `BatchEvalPython` and may expose Python worker transfer metrics, including data sent to and returned from Python workers.


In [4]:
base_stream = (
    spark.range(0, 1_000_000, 1, numPartitions=32)
    .withColumn("sector_id", (F.col("id") % 500).cast("int"))
)

spark.sparkContext.setJobDescription("🟢 Target Gamma-1: Native Spark Expression")

native_stream = base_stream.withColumn(
    "tier_class",
    F.when(F.col("sector_id") % 2 == 0, F.lit("PRIME")).otherwise(F.lit("STANDARD"))
)

print("🧱 Native expression plan:")
native_stream.groupBy("tier_class").count().explain("formatted")

start_clock = time.time()
native_result = native_stream.groupBy("tier_class").count().collect()
end_clock = time.time()

print(f"🎯 Native expression result: {native_result}")
print(f"⏱️ Native duration: {end_clock - start_clock:.2f} seconds.")

spark.sparkContext.setJobDescription("🐍 Target Gamma-2: Python UDF Boundary")

@F.udf(StringType())
def classify_sector_tier(sector):
    if sector is None:
        return "VOID"
    return "PRIME" if sector % 2 == 0 else "STANDARD"

boundary_stream = base_stream.withColumn(
    "tier_class",
    classify_sector_tier(F.col("sector_id"))
)

print("🧱 Python UDF plan:")
boundary_stream.groupBy("tier_class").count().explain("formatted")

start_clock = time.time()
python_result = boundary_stream.groupBy("tier_class").count().collect()
end_clock = time.time()

print(f"🎯 Python UDF result: {python_result}")
print(f"⏱️ Python UDF duration: {end_clock - start_clock:.2f} seconds.")
print("Open the SQL tab and compare the native plan with the Python UDF plan. Look for BatchEvalPython/Python worker metrics when your Spark UI exposes them.")


🧱 Native expression plan:
== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Project (2)
            +- Range (1)


(1) Range
Output [1]: [id#21L]
Arguments: Range (0, 1000000, step=1, splits=Some(32))

(2) Project
Output [1]: [CASE WHEN ((cast((id#21L % 500) as int) % 2) = 0) THEN PRIME ELSE STANDARD END AS tier_class#23]
Input [1]: [id#21L]

(3) HashAggregate
Input [1]: [tier_class#23]
Keys [1]: [tier_class#23]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#29L]
Results [2]: [tier_class#23, count#30L]

(4) Exchange
Input [2]: [tier_class#23, count#30L]
Arguments: hashpartitioning(tier_class#23, 100), ENSURE_REQUIREMENTS, [plan_id=217]

(5) HashAggregate
Input [2]: [tier_class#23, count#30L]
Keys [1]: [tier_class#23]
Functions [1]: [count(1)]
Aggregate Attributes [1]: [count(1)#28L]
Results [2]: [tier_class#23, count(1)#28L AS count#24L]

(6) AdaptiveSparkPlan
Output [2]: [tier_class#23, count#24

### Step 5: Trigger Strict ANSI Mode Red Alert Failure

We intentionally cast malformed text to an integer. With ANSI mode enabled, Spark should fail loudly instead of quietly returning `NULL` for the malformed value.

This is expected. The goal is to create a failed SQL/DataFrame execution that you can inspect in the SQL tab.


In [5]:
spark.sparkContext.setJobDescription("🚨 Target Delta: Intentional ANSI Cast Failure")

corrupted_payload = spark.createDataFrame(
    [("MAD_QUADRANT",), ("12345",)],
    ["raw_string_coordinate"]
)

failed_cast_query = corrupted_payload.withColumn(
    "parsed_integer",
    F.col("raw_string_coordinate").cast("int")
)

try:
    print("🔥 Launching broken data payload through strict ANSI conversion path...")
    failed_cast_query.collect()
except Exception as e:
    print("💥 Expected ANSI failure captured.")
    print(type(e).__name__)
    print(str(e)[:1000])

print("Open the SQL tab and inspect the failed execution entry and error message.")


🔥 Launching broken data payload through strict ANSI conversion path...


{"ts": "2026-07-07 19:06:20.021", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value 'MAD_QUADRANT' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 10 in cell [6]", "line": "", "fragment": "cast", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o171.collectToPython.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value 'MAD_QUADRANT' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"cast\" was called from\nline 10 in cell [6]\n\r\n\tat org.apache.spark.sql.erro

💥 Expected ANSI failure captured.
NumberFormatException
[CAST_INVALID_INPUT] The value 'MAD_QUADRANT' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 10 in cell [6]

Open the SQL tab and inspect the failed execution entry and error message.


### Step 6: Keep the SQL Targeting Suite Open

We keep the driver process alive so the Live UI remains available long enough to inspect the SQL tab, associated jobs, physical plans, operator metrics, and the failed query.


In [6]:
print(f"⏳ Explore the SQL tab now: {spark.sparkContext.uiWebUrl}")
print("Suggested checks:")
print("1. Target Alpha: adaptive/final plan, exchanges, query stages, possible coalesced shuffle readers.")
print("2. Target Beta: join output rows and row explosion symptoms.")
print("3. Target Gamma: native expression vs Python UDF plan and Python worker transfer metrics if exposed.")
print("4. Target Delta: failed execution and ANSI error details.")

input("Press Enter when you are done inspecting the SQL tab and want to stop Spark...")

spark.stop()
print("💀 SparkContext stopped. The Live UI is no longer available.")


⏳ Explore the SQL tab now: http://T14-PF4WM3XL.sdggroup.local:4040
Suggested checks:
1. Target Alpha: adaptive/final plan, exchanges, query stages, possible coalesced shuffle readers.
2. Target Beta: join output rows and row explosion symptoms.
3. Target Gamma: native expression vs Python UDF plan and Python worker transfer metrics if exposed.
4. Target Delta: failed execution and ANSI error details.
💀 SparkContext stopped. The Live UI is no longer available.


## 📊 Post-Lab Analysis: Evidence from the Torpedo Room

### 1. AQE Is Evidence to Verify, Not a Spell to Assume

Step 2 creates a shuffle workload where AQE may coalesce post-shuffle partitions after runtime statistics are available.

The correct lesson is not "AQE definitely merged everything." The correct lesson is: inspect the SQL tab or final adaptive physical plan and verify what Spark actually changed.

### 2. Row Explosion Shows Up in Operator Metrics

Step 3 creates a many-to-many join. The joined output is much larger than either input.

In the SQL tab, the join operator's output-row metric is the lie detector. If a join receives modest inputs and emits millions or billions of rows, the problem is not that Spark is mysteriously slow. The problem is that your join multiplied the quadrant.

### 3. Python UDFs Are Border Crossings

Step 4 compares a native Spark SQL expression with a row-at-a-time Python UDF.

The Python UDF path may show Python execution operators such as `BatchEvalPython`, plus metrics for data sent to and returned from Python workers when your Spark version and UI expose them. These metrics are byte/transfer signals, not row counts.

### 4. ANSI Mode Turns Bad Data into Loud Failures

Step 5 verifies ANSI behavior explicitly by enabling `spark.sql.ansi.enabled` and casting malformed text to an integer.

The failure is intentional. It should appear as a failed SQL/DataFrame execution, giving you a clean place to inspect error details instead of letting bad data quietly become nulls downstream.

### ⏱️ Validation Summary

* **Step 1:** Started a local Spark session, explicitly enabled AQE and ANSI mode, and printed the actual Spark UI URL.
* **Step 2:** Created a shuffle workload where AQE may adapt the final physical plan.
* **Step 3:** Triggered a controlled join row explosion and inspected join/output metrics.
* **Step 4:** Compared native Spark expressions with a Python UDF boundary and checked for Python worker transfer metrics.
* **Step 5:** Triggered and captured an intentional ANSI cast failure.
* **Step 6:** Kept the Live UI open long enough to inspect SQL executions, plans, metrics, and failed-query diagnostics.
